# Step 10 — Heston Model via Douglas ADI

The paper underlying this repository — *von Sydow et al. (2010), "ADI finite difference schemes for option pricing in the Heston model with correlation"* — is specifically about the **Heston stochastic-volatility model**, not SABR. Notebook 07 applied the same Douglas-scheme ADI technique to SABR; this notebook prices options under the model the paper actually studies.

## Heston dynamics

$$dS = r S\, dt + \sqrt{v}\, S\, dW_1$$
$$dv = \kappa(\theta - v)\, dt + \xi\sqrt{v}\, dW_2, \qquad dW_1\, dW_2 = \rho\, dt$$

| Parameter | Meaning |
|-----------|--------|
| $\kappa$ | Mean-reversion speed of variance |
| $\theta$ | Long-run variance (so $\sqrt{\theta}$ is the long-run vol) |
| $\xi$ | Vol-of-vol |
| $\rho$ | Spot–vol correlation (negative for equity skew) |
| $v_0$ | Initial variance |

## PDE

Under the risk-neutral measure the option price $V(S, v, t)$ satisfies

$$\frac{\partial V}{\partial t} + \underbrace{\frac{1}{2}v S^2\frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - \frac{r}{2}V}_{F_1} + \underbrace{\rho\xi v S\frac{\partial^2 V}{\partial S\,\partial v}}_{F_0} + \underbrace{\frac{1}{2}\xi^2 v\frac{\partial^2 V}{\partial v^2} + \kappa(\theta-v)\frac{\partial V}{\partial v} - \frac{r}{2}V}_{F_2} = 0$$

Compared with the SABR PDE in notebook 07, the differences are: (i) the S-diffusion uses $v$ instead of $v^{2\beta}$; (ii) $F_2$ gains the mean-reversion drift $\kappa(\theta-v)\partial V/\partial v$; (iii) $F_0$ uses $\xi v$ instead of $\nu v^2$.

## Douglas ADI scheme

Identical operator-splitting structure as notebook 07 — $F_0$ explicit, $F_1$ implicit in $S$, $F_2$ implicit in $v$:

$$Y_0 = V^n + \Delta\tau\,(F_0 + F_1 + F_2)V^n$$
$$(I - \theta\Delta\tau F_1)Y_1 = Y_0 - \theta\Delta\tau F_1 V^n$$
$$(I - \theta\Delta\tau F_2)V^{n+1} = Y_1 - \theta\Delta\tau F_2 V^n$$

$\theta = 1/2$ gives second-order accuracy.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.stats import norm

from finite_difference import bs_price

## Parameters

In [ ]:
# Heston model
kappa = 2.0     # mean-reversion speed
theta = 0.04    # long-run variance  (long-run vol = 20%)
xi    = 0.3     # vol-of-vol
rho   = -0.7    # spot-vol correlation (negative for equity skew)
v0    = 0.04    # initial variance    (initial vol = 20%)

# Option
S0    = 100.0
K     = 100.0
r     = 0.05
T     = 1.0

# Feller condition (2κθ >= ξ² keeps v > 0)
print(f'Feller condition: 2κθ = {2*kappa*theta:.4f},  ξ² = {xi**2:.4f}',
      '  SATISFIED' if 2*kappa*theta >= xi**2 else '  VIOLATED')

## 1. Heston (1993) Analytical Price

The Heston model has a semi-closed-form price via Fourier inversion (Heston 1993):

$$C = S_0 P_1 - K e^{-rT} P_2$$

where $P_1, P_2$ are risk-neutral probabilities recovered by inverting two characteristic functions:

$$P_j = \frac{1}{2} + \frac{1}{\pi}\int_0^\infty \mathrm{Re}\!\left[\frac{e^{-i\phi\ln K}\, f_j(\phi)}{i\phi}\right] d\phi$$

The implementation below uses the **Albrecher et al. (2007)** parameterisation, which avoids the branch-cut discontinuity of the original Heston (1993) characteristic function.

In [ ]:
def heston_char(phi, j, S0, K, r, T, kappa, theta, xi, rho, v0):
    """Heston characteristic function f_j at frequency phi (Albrecher 2007)."""
    i = 1j
    u = 0.5  if j == 1 else -0.5
    b = kappa - rho * xi  if j == 1 else kappa

    x   = b - rho * xi * i * phi
    d   = np.sqrt(x**2 + xi**2 * (phi**2 - 2.0 * u * i * phi))
    g   = (x - d) / (x + d)          # Albrecher form; avoids branch cut
    edT = np.exp(-d * T)

    C = r * i * phi * T + (kappa * theta / xi**2) * (
        (x - d) * T - 2.0 * np.log((1.0 - g * edT) / (1.0 - g))
    )
    D = (x - d) / xi**2 * (1.0 - edT) / (1.0 - g * edT)

    return np.exp(C + D * v0 + i * phi * (np.log(S0 / K) + r * T))


def heston_call(S0, K, r, T, kappa, theta, xi, rho, v0):
    """European call price under Heston (1993) via Fourier inversion."""
    def integrand(phi, j):
        return np.real(heston_char(phi, j, S0, K, r, T, kappa, theta, xi, rho, v0) / (1j * phi))

    I1 = quad(lambda p: integrand(p, 1), 1e-9, 200, limit=300, epsabs=1e-9)[0]
    I2 = quad(lambda p: integrand(p, 2), 1e-9, 200, limit=300, epsabs=1e-9)[0]

    P1 = 0.5 + I1 / np.pi
    P2 = 0.5 + I2 / np.pi
    return S0 * P1 - K * np.exp(-r * T) * P2


analytical = heston_call(S0, K, r, T, kappa, theta, xi, rho, v0)
bs_equiv   = bs_price(S0, K, T, r, np.sqrt(theta), option_type='call')  # BS at long-run vol
print(f'Heston call (analytical): {analytical:.4f}')
print(f'BS call at sqrt(theta)=20% vol: {bs_equiv:.4f}')

## 2. Douglas ADI Solver for Heston

The implementation mirrors the library's `ADISolver` in `finite_difference/solvers/adi.py` but with Heston-specific operators. Key differences from SABR:

| | SABR | Heston |
|---|---|---|
| S-diffusion $a_{ij}$ | $\frac{1}{2}v_j^2 S_i^{2\beta}$ | $\frac{1}{2}v_j S_i^2$ |
| v-drift | — | $\kappa(\theta - v_j)\,\partial V/\partial v$ |
| v-diffusion $c_j$ | $\frac{1}{2}\nu^2 v_j^2$ | $\frac{1}{2}\xi^2 v_j$ |
| Mixed $\gamma_{ij}$ | $\rho\nu v_j^2 S_i^\beta$ | $\rho\xi v_j S_i$ |

In [ ]:
def _thomas_batch(sub, mid, sup, rhs):
    """
    Thomas algorithm for a batch of tridiagonal systems.
    sub, mid, sup: (n,) or (n, batch) — identical to the library's _thomas_batch.
    rhs: (n, batch). Returns solution of same shape.
    """
    n   = mid.shape[0]
    cp  = np.empty_like(mid, dtype=float)
    dp  = np.empty_like(rhs, dtype=float)

    cp[0] = sup[0] / mid[0]
    dp[0] = rhs[0] / mid[0]
    for k in range(1, n):
        denom = mid[k] - sub[k] * cp[k - 1]
        cp[k] = sup[k] / denom
        dp[k] = (rhs[k] - sub[k] * dp[k - 1]) / denom

    x     = np.empty_like(rhs, dtype=float)
    x[-1] = dp[-1]
    for k in range(n - 2, -1, -1):
        x[k] = dp[k] - cp[k] * x[k + 1]
    return x


class HestonADI:
    """
    Douglas-scheme ADI solver for European options under the Heston model.
    Grid: S in [0, S_max], v in [0, v_max].
    """

    def __init__(self, kappa, theta, xi, rho, r, K, T,
                 S_max, v_max, M, L, N, option_type='call'):
        self.kappa, self.theta, self.xi, self.rho = kappa, theta, xi, rho
        self.r, self.K, self.T = r, K, T
        self.M, self.L, self.N = M, L, N
        self.option_type = option_type

        self.S  = np.linspace(0.0, S_max, M + 1)
        self.v  = np.linspace(0.0, v_max, L + 1)
        self.dS = S_max / M
        self.dv = v_max / L
        self.dt = T / N

        # Precompute fixed coefficient arrays
        Si    = self.S[1:M, None]          # (M-1, 1)
        vj    = self.v[None, 1:L]          # (1, L-1)

        self._aS    = 0.5 * vj * Si**2                        # (M-1, L-1)
        self._bS    = self.r * self.S[1:M]                    # (M-1,)
        self._aV    = 0.5 * xi**2 * self.v[1:L]               # (L-1,)
        self._drift = kappa * (theta - self.v[1:L])           # (L-1,)
        self._gamma = rho * xi * vj * Si                      # (M-1, L-1)

    # ------------------------------------------------------------------
    # Boundary conditions
    # ------------------------------------------------------------------

    def _apply_bc(self, V, tau):
        df = np.exp(-self.r * tau)
        S  = self.S
        if self.option_type == 'call':
            V[0, :]         = 0.0
            V[self.M, :]    = S[self.M] - self.K * df
            V[:, 0]         = np.maximum(S - self.K * df, 0.0)   # v=0: BS with σ=0
        else:
            V[0, :]         = self.K * df
            V[self.M, :]    = 0.0
            V[:, 0]         = np.maximum(self.K * df - S, 0.0)
        V[:, self.L] = V[:, self.L - 1]                          # Neumann at v_max
        return V

    # ------------------------------------------------------------------
    # Spatial operators
    # ------------------------------------------------------------------

    def _F0(self, V):
        """Mixed derivative (explicit)."""
        out = np.zeros_like(V)
        out[1:-1, 1:-1] = (
            self._gamma
            * (V[2:, 2:] - V[2:, :-2] - V[:-2, 2:] + V[:-2, :-2])
            / (4.0 * self.dS * self.dv)
        )
        return out

    def _F1(self, V):
        """S-direction operator."""
        out = np.zeros_like(V)
        a   = self._aS / self.dS**2          # (M-1, L-1)
        b   = self._bS[:, None] / (2.0 * self.dS)
        out[1:-1, 1:-1] = (
            (a - b) * V[:-2, 1:-1]
            + (-2.0 * a - 0.5 * self.r) * V[1:-1, 1:-1]
            + (a + b) * V[2:,  1:-1]
        )
        return out

    def _F2(self, V):
        """v-direction operator (includes mean-reversion drift)."""
        out = np.zeros_like(V)
        c   = self._aV[None, :] / self.dv**2          # (1, L-1)
        d   = self._drift[None, :] / (2.0 * self.dv)  # (1, L-1)
        out[1:-1, 1:-1] = (
            (c - d) * V[1:-1, :-2]
            + (-2.0 * c - 0.5 * self.r) * V[1:-1, 1:-1]
            + (c + d) * V[1:-1, 2:]
        )
        return out

    # ------------------------------------------------------------------
    # Implicit sweeps
    # ------------------------------------------------------------------

    def _solve_S_implicit(self, rhs, theta_dt, tau):
        M, L = self.M, self.L
        Y    = self._apply_bc(rhs.copy(), tau)

        a   = self._aS / self.dS**2                      # (M-1, L-1)
        b   = self._bS[:, None] / (2.0 * self.dS)        # (M-1, 1) → broadcasts

        sub = -theta_dt * (a - b)                         # (M-1, L-1)
        mid =  1.0 + theta_dt * (2.0 * a + 0.5 * self.r)
        sup = -theta_dt * (a + b)

        rhs_int = rhs[1:M, 1:L].copy()
        rhs_int[0,  :] -= sub[0,  :] * Y[0,  1:L]   # S=0 BC
        rhs_int[-1, :] -= sup[-1, :] * Y[M,  1:L]   # S=S_max BC

        Y[1:M, 1:L] = _thomas_batch(sub, mid, sup, rhs_int)
        return self._apply_bc(Y, tau)

    def _solve_v_implicit(self, rhs, theta_dt, tau):
        M, L = self.M, self.L
        Y    = self._apply_bc(rhs.copy(), tau)

        c   = self._aV / self.dv**2                       # (L-1,)
        d   = self._drift / (2.0 * self.dv)               # (L-1,)

        sub = -theta_dt * (c - d)
        mid =  1.0 + theta_dt * (2.0 * c + 0.5 * self.r)
        sup = -theta_dt * (c + d)

        # Neumann at v_max: fold last super-diagonal into diagonal
        mid_eff     = mid.copy()
        mid_eff[-1] = mid[-1] + sup[-1]

        rhs_int = rhs[:, 1:L].T.copy()           # (L-1, M+1)
        rhs_int[0, :] -= sub[0] * Y[:, 0]        # v=0 Dirichlet

        sol        = _thomas_batch(sub, mid_eff, sup, rhs_int)  # (L-1, M+1)
        Y[:, 1:L]  = sol.T
        Y[:, L]    = Y[:, L - 1]                  # mirror Neumann
        return self._apply_bc(Y, tau)

    # ------------------------------------------------------------------
    # Time stepping
    # ------------------------------------------------------------------

    def solve(self, theta=0.5, verbose=False):
        """Advance from tau=0 (expiry) to tau=T (today) and return (V, S, v)."""
        V      = np.maximum(self.S[:, None] - self.K, 0.0) * np.ones((1, self.L + 1))
        V      = self._apply_bc(V.copy(), 0.0)

        for n in range(self.N):
            tau_new = (n + 1) * self.dt
            tau_old = n       * self.dt
            dt      = self.dt

            F0V = self._F0(V)
            F1V = self._F1(V)
            F2V = self._F2(V)

            Y0 = self._apply_bc(V + dt * (F0V + F1V + F2V), tau_new)
            Y1 = self._solve_S_implicit(Y0 - theta * dt * F1V, theta * dt, tau_new)
            V  = self._solve_v_implicit(Y1 - theta * dt * F2V, theta * dt, tau_new)

            if verbose and (n % max(1, self.N // 5) == 0):
                mid = V[self.M // 2, self.L // 4]
                print(f'  step {n+1:3d}/{self.N}  tau={tau_new:.3f}  V[mid]={mid:.4f}')

        return V, self.S, self.v

## 3. Validation Against Analytical Price

In [ ]:
S_max = 4.0 * S0
v_max = 0.5           # variance up to 50% → vol up to ~70%
M, L, N = 80, 40, 80

solver = HestonADI(kappa, theta, xi, rho, r, K, T,
                   S_max, v_max, M, L, N, option_type='call')

t0 = time.time()
V, S_grid, v_grid = solver.solve(theta=0.5, verbose=True)
elapsed = time.time() - t0
print(f'Solved in {elapsed:.2f}s')

# Bilinear interpolation at (S0, v0)
i0  = int(np.searchsorted(S_grid, S0))
j0  = int(np.searchsorted(v_grid, v0))
i0  = max(1, min(i0, M))
j0  = max(1, min(j0, L))
wS  = (S0 - S_grid[i0-1]) / (S_grid[i0] - S_grid[i0-1])
wv  = (v0 - v_grid[j0-1]) / (v_grid[j0] - v_grid[j0-1])
fd_price = (
    (1-wS)*(1-wv)*V[i0-1,j0-1] + wS*(1-wv)*V[i0,j0-1]
  + (1-wS)*wv   *V[i0-1,j0]   + wS*wv    *V[i0,j0]
)

print(f'\nHeston call (FD, M=L={M}, N={N}): {fd_price:.4f}')
print(f'Heston call (analytical):          {analytical:.4f}')
print(f'Absolute error:                    {abs(fd_price - analytical):.4f}')

assert abs(fd_price - analytical) < 0.15, 'FD price deviates too far from analytical'

## 4. Price Surface

In [ ]:
S_plot = S_grid[S_grid <= 200]
v_plot = v_grid[v_grid <= 0.3]
S_mesh, v_mesh = np.meshgrid(S_plot, v_plot, indexing='ij')
V_plot = V[:len(S_plot), :len(v_plot)]

fig = plt.figure(figsize=(14, 4))

# 3-D surface
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(S_mesh, np.sqrt(v_mesh), V_plot,
                 cmap='viridis', alpha=0.85, linewidth=0)
ax1.set_xlabel('S')
ax1.set_ylabel('σ = √v')
ax1.set_zlabel('V')
ax1.set_title('Heston Call Surface')

# Slice: V vs S at v = v0
ax2 = fig.add_subplot(132)
j_v0  = int(np.searchsorted(v_grid, v0))
S_bs  = np.linspace(50, 200, 300)
V_bs  = np.array([bs_price(s, K, T, r, np.sqrt(v0), 'call') for s in S_bs])
ax2.plot(S_plot, V_plot[:, j_v0], label='Heston FD')
ax2.plot(S_bs, V_bs, '--', label=f'BS at σ={np.sqrt(v0):.0%}')
ax2.axvline(K, color='k', lw=0.8, ls='--', alpha=0.3)
ax2.set_xlabel('S')
ax2.set_ylabel('Call value')
ax2.set_title(f'V vs S  (v = v₀ = {v0})')
ax2.legend(fontsize=8)
ax2.grid(True, ls='--', alpha=0.4)

# Slice: V vs v at S = S0
ax3 = fig.add_subplot(133)
i_S0  = int(np.searchsorted(S_grid, S0))
ax3.plot(np.sqrt(v_plot), V_plot[i_S0, :], label='Heston FD')
ax3.set_xlabel('σ = √v')
ax3.set_ylabel('Call value')
ax3.set_title(f'V vs σ  (S = S₀ = {S0})')
ax3.grid(True, ls='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../results/10_heston_surface.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Implied-Volatility Smile

The defining feature of the Heston model is that it generates a curved implied-volatility smile — unlike Black–Scholes, which implies a flat surface by construction. The negative correlation $\rho < 0$ creates a **skew**: downside strikes carry higher implied vol than upside strikes, consistent with observed equity markets.

In [ ]:
def implied_vol(price, S0, K, T, r, option_type='call'):
    """Invert Black-Scholes for implied vol via Brent's method."""
    intrinsic = max(S0 - K * np.exp(-r * T), 0.0) if option_type == 'call' else max(K * np.exp(-r * T) - S0, 0.0)
    if price <= intrinsic + 1e-8:
        return np.nan
    try:
        return brentq(lambda s: bs_price(S0, K, T, r, s, option_type) - price,
                      1e-4, 5.0, xtol=1e-6)
    except ValueError:
        return np.nan


strikes = np.linspace(70, 140, 25)
iv_heston = []
iv_bs_flat = []

for Ki in strikes:
    price_an = heston_call(S0, Ki, r, T, kappa, theta, xi, rho, v0)
    iv_heston.append(implied_vol(price_an, S0, Ki, T, r, 'call'))
    iv_bs_flat.append(np.sqrt(theta))  # constant BS vol = long-run vol

iv_heston = np.array(iv_heston)

plt.figure(figsize=(8, 4))
plt.plot(strikes, iv_heston * 100, 'o-', label='Heston implied vol')
plt.axhline(np.sqrt(theta) * 100, color='gray', ls='--',
            label=f'BS flat vol = √θ = {np.sqrt(theta):.0%}')
plt.axvline(K, color='k', lw=0.8, ls='--', alpha=0.3, label='ATM')
plt.xlabel('Strike K')
plt.ylabel('Implied volatility (%)')
plt.title(f'Heston Implied-Volatility Smile  (ρ={rho}, ξ={xi})')
plt.legend()
plt.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../results/10_heston_smile.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'ATM implied vol:    {implied_vol(heston_call(S0, K, r, T, kappa, theta, xi, rho, v0), S0, K, T, r):.2%}')
print(f'OTM put (K=80) IV:  {iv_heston[np.argmin(np.abs(strikes - 80))]:.2%}')
print(f'OTM call (K=130) IV:{iv_heston[np.argmin(np.abs(strikes - 130))]:.2%}')

## 6. Convergence

In [ ]:
ref = analytical
print(f'Analytical reference: {ref:.6f}')

configs = [(20, 10, 20), (40, 20, 40), (60, 30, 60), (80, 40, 80), (100, 50, 100)]
errors  = []

for Mg, Lg, Ng in configs:
    sol = HestonADI(kappa, theta, xi, rho, r, K, T,
                    S_max, v_max, Mg, Lg, Ng, option_type='call')
    Vg, Sg, vg = sol.solve(theta=0.5)

    i0 = max(1, min(int(np.searchsorted(Sg, S0)), Mg))
    j0 = max(1, min(int(np.searchsorted(vg, v0)), Lg))
    wS = (S0 - Sg[i0-1]) / (Sg[i0] - Sg[i0-1])
    wv = (v0 - vg[j0-1]) / (vg[j0] - vg[j0-1])
    p  = ((1-wS)*(1-wv)*Vg[i0-1,j0-1] + wS*(1-wv)*Vg[i0,j0-1]
         + (1-wS)*wv   *Vg[i0-1,j0]   + wS*wv    *Vg[i0,j0])

    err = abs(p - ref)
    errors.append(err)
    print(f'M={Mg:3d} L={Lg:2d} N={Ng:3d}  price={p:.5f}  error={err:.5f}')

sizes = [c[0] for c in configs]
plt.figure(figsize=(7, 4))
plt.loglog(sizes, errors, 'o-', label='FD error vs analytical')
plt.loglog(sizes,
           [errors[0] * (sizes[0] / s)**2 for s in sizes],
           '--', color='gray', alpha=0.7, label=r'$O(M^{-2})$ reference')
plt.xlabel('Grid size M')
plt.ylabel('Absolute error')
plt.title('Heston ADI Convergence (θ = ½)')
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../results/10_heston_convergence.png', dpi=150, bbox_inches='tight')
plt.show()